# RLT Enhanced Quick Start
## Production-Ready Implementation with Claude Sonnet 4

This notebook demonstrates the enhanced RLT system with:
- Claude API integration for high-quality explanations
- Cost tracking and budget management
- Multi-dataset support
- Dense reward computation
- Efficient caching and error handling

## 1. Setup and Installation

In [ ]:
# Install required packages
!pip install -q anthropic transformers torch datasets accelerate
!pip install -q python-dotenv tqdm matplotlib seaborn

# Setup project structure
import os
import sys

# Add src to path
if 'src' not in sys.path:
    sys.path.insert(0, os.path.abspath('.'))

print("✅ Environment setup complete")

## 2. Configure API Keys

In [ ]:
# Configure API key (for Google Colab)
import os
from getpass import getpass

# Option 1: Environment variable
if 'CLAUDE_API_KEY' not in os.environ:
    print("Please enter your Claude API key:")
    os.environ['CLAUDE_API_KEY'] = getpass()

# Option 2: Load from .env file
try:
    from dotenv import load_dotenv
    load_dotenv()
except:
    pass

# Verify
if os.environ.get('CLAUDE_API_KEY'):
    print("✅ API key configured")
else:
    print("❌ API key not found")

## 3. Initialize Enhanced Components

In [ ]:
from src.utils.cost_tracker import CostTracker
from src.teachers.claude_teacher import ClaudeRLTTeacher
from src.data import DataLoader, DataProcessor
from src.rewards import RLTRewardFunction, LocalStudentEvaluator, TransformerKLCalculator

# Initialize cost tracker with budget
cost_tracker = CostTracker(budget_limit=5.0)  # $5 budget for demo

# Initialize Claude teacher
teacher = ClaudeRLTTeacher(
    api_key=os.environ.get('CLAUDE_API_KEY'),
    cost_tracker=cost_tracker,
    cache_enabled=True
)

# Initialize data components
data_loader = DataLoader()
data_processor = DataProcessor()

# Initialize reward components
student_evaluator = LocalStudentEvaluator(model_name="microsoft/phi-2")
kl_calculator = TransformerKLCalculator(model_name="gpt2")
reward_function = RLTRewardFunction(student_evaluator, kl_calculator)

print("✅ All components initialized")

## 4. Load Sample Data

In [ ]:
# Load a small sample from GSM8K
print("Loading GSM8K dataset...")
dataset = data_loader.load_dataset('gsm8k', split='train', max_samples=5)

# Display sample
print(f"\nLoaded {len(dataset)} samples")
print("\nSample problem:")
print(f"Question: {dataset[0].question}")
print(f"Answer: {dataset[0].answer}")
print(f"Subject: {dataset[0].subject}")
print(f"Difficulty: {dataset[0].difficulty}")

## 5. Generate Teacher Explanations with Claude

In [ ]:
# Generate explanations for first few samples
explanations = []
rewards = []

print("Generating explanations with Claude...\n")

for i, item in enumerate(dataset[:3]):  # Only 3 for demo
    print(f"\n{'='*60}")
    print(f"Sample {i+1}:")
    print(f"Question: {item.question[:100]}...")
    
    # Generate explanation
    result = teacher.generate_explanation(
        question=item.question,
        answer=item.answer,
        temperature=0.7
    )
    
    explanation = result['explanation']
    explanations.append(explanation)
    
    print(f"\nExplanation preview: {explanation[:200]}...")
    print(f"Cost: ${result['usage']['total_cost']:.4f}")
    
    # Compute reward
    reward_result = reward_function.compute_reward(
        [explanation],
        [item.question]
    )
    rewards.append(reward_result['rewards'][0])
    
    print(f"\nReward: {reward_result['rewards'][0]:.4f}")
    print(f"rSS: {reward_result['rSS'][0]:.4f}")
    print(f"rKL: {reward_result['rKL'][0]:.4f}")

# Show cost summary
print(f"\n{'='*60}")
print("Cost Summary:")
for key, value in cost_tracker.get_summary().items():
    print(f"{key}: {value}")

## 6. Visualize Results

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

if len(rewards) > 0:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    
    # Reward distribution
    ax1.bar(range(len(rewards)), rewards)
    ax1.set_xlabel('Sample')
    ax1.set_ylabel('Reward')
    ax1.set_title('Reward Distribution')
    ax1.axhline(y=np.mean(rewards), color='r', linestyle='--', label=f'Mean: {np.mean(rewards):.3f}')
    ax1.legend()
    
    # Cost progression
    history = cost_tracker.costs['history']
    if history:
        cumulative_costs = [h['cumulative_cost'] for h in history]
        ax2.plot(cumulative_costs, 'b-', marker='o')
        ax2.set_xlabel('API Call')
        ax2.set_ylabel('Cumulative Cost ($)')
        ax2.set_title('API Cost Progression')
        ax2.axhline(y=cost_tracker.budget_limit, color='r', linestyle='--', label=f'Budget: ${cost_tracker.budget_limit}')
        ax2.legend()
    
    plt.tight_layout()
    plt.show()
else:
    print("No rewards to visualize")

## 7. Create Distillation Dataset

In [ ]:
# Create training data for student model
distillation_data = []

for i, (item, explanation, reward) in enumerate(zip(dataset[:3], explanations, rewards)):
    student_input = data_processor.format_student_input(item.question, explanation)
    
    distillation_data.append({
        'input': student_input,
        'output': item.answer,
        'reward': reward,
        'metadata': {
            'subject': item.subject,
            'difficulty': item.difficulty
        }
    })

# Save distillation dataset
import json
with open('distillation_data.json', 'w') as f:
    json.dump(distillation_data, f, indent=2)

print(f"✅ Created {len(distillation_data)} training examples")
print("\nExample student training data:")
print(f"Input: {distillation_data[0]['input'][:200]}...")
print(f"Output: {distillation_data[0]['output']}")
print(f"Reward: {distillation_data[0]['reward']:.4f}")

## 8. Next Steps

### To scale this to production:

1. **Increase Dataset Size**: Load more samples from GSM8K, MATH, or ARC-C
2. **Batch Processing**: Use `teacher.batch_generate_explanations()` for efficiency
3. **GRPO Training**: Implement the full GRPO training loop
4. **Student Fine-tuning**: Train student model on high-reward explanations
5. **Evaluation**: Use the evaluation framework on benchmarks

### Key Features Demonstrated:
- ✅ Claude API integration with cost tracking
- ✅ Multi-dataset support with caching
- ✅ Dense reward computation (rSS - λ*rKL)
- ✅ Production-ready error handling
- ✅ Visualization and monitoring

### Recommended Budget:
- Development/Testing: $25-50
- Full Training Run: $100-500 (depending on dataset size)
- Use caching aggressively to minimize costs!